### 1.2 Формулировка

**Входы**

- Задание и результаты первого знакомства с данными

**Выходы**

- Что предсказываем, по каким данным и для кого
- Требования к данным
- Полный набор метрик качества и правила выбора модели и порога

**Что делаем**

Собираем постановку задачи: что подаём модели, что хотим получить и как поймём, насколько хорошо она работает

In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


##### Для кого строим прогноз

#### Как будем понимать задачу

Предположим, что заёмщик уже подал заявку на кредит. Кредитор оценил её, назначил грейд `loan_grade` и предложил ставку `loan_int_rate`

Наша модель нужна инвестору, который решает, финансировать ли такой кредит. Поэтому грейд и ставку можно использовать: в выбранный момент они уже известны. Возможно, при своей оценке кредитор учёл больше сведений, чем есть у нас. Это допущение о сценарии работы, а не установленный факт об источнике данных

Столбец `loan_status` показывает фактический исход: 0 — без дефолта, 1 — дефолт. Будем предсказывать его по остальным столбцам. Столбец, который предсказываем, называют **целевой переменной**

Задача относится к **бинарной классификации** — предсказанию одного из двух классов. Модель выдаёт оценку риска, которую по выбранному **порогу** переводим в класс 0 или 1

#### Что подаём модели и что получаем

Разметка — это известные исходы прошлых кредитов в столбце `loan_status`. По ним модель учится отличать дефолты от остальных случаев

| Что определяем | В нашем случае |
| --- | --- |
| Одна запись | Заявка на кредит |
| Вход | Все 11 признаков, кроме `loan_status` |
| Результат | Оценка риска, а после применения порога — класс 0 или 1 |
| Кто использует прогноз | Инвестор, который рассматривает финансирование кредита |
| Когда строим прогноз | Грейд и ставка уже назначены, фактический исход ещё неизвестен |

Алгоритм пока не выбираем: сначала разберём данные

#### Какие данные нужны

Для обучения нужен известный исход кредита — `loan_status`. Сам исход не подаём модели как признак

Для новых заявок нужны те же 11 полей в исходных единицах измерения

Используем только то, что известно в момент прогноза. В нашем сценарии грейд и ставка уже есть, а фактического исхода кредита ещё нет. Прошлые дефолты должны относиться именно к прошлому, а не к будущим событиям

Пропуски, повторы и подозрительные значения проверим при EDA. Период, за который автор определял дефолт, не указан: модель будет предсказывать метку в том виде, в котором она дана в источнике

#### Критерии оценивания

Набор метрик задаём для задачи бинарной классификации и используем для всех моделей независимо от их устройства. Положительный класс — дефолт: `loan_status = 1`

| Обозначение | Число кредитов |
| --- | --- |
| **TP** | Верно предсказан дефолт |
| **FP** | Предсказан дефолт, фактически его нет |
| **FN** | Дефолт не предсказан, фактически он произошёл |
| **TN** | Верно предсказано отсутствие дефолта |

##### Метрики по предсказанным классам

| Метрика | Формула | Что показывает |
| --- | --- | --- |
| Accuracy | $\frac{TP+TN}{TP+FP+FN+TN}$ | Доля верных ответов |
| Precision (точность) | $\frac{TP}{TP+FP}$ | Доля настоящих дефолтов среди предсказанных |
| Recall (полнота) | $\frac{TP}{TP+FN}$ | Доля обнаруженных дефолтов среди всех фактических |
| F1 | $\frac{2TP}{2TP+FP+FN}$ | Объединяет precision и recall с равными весами |
| F2 | $\frac{5TP}{5TP+FP+4FN}$ | Объединяет precision и recall с большим весом recall |
| Balanced accuracy | $\frac{1}{2}\left(\frac{TP}{TP+FN}+\frac{TN}{TN+FP}\right)$ | Средняя полнота по двум классам |

Если знаменатель precision, recall, F1 или F2 равен нулю, принимаем значение метрики равным 0. Для всех метрик этой группы больше — лучше

##### Метрики по оценкам риска

**Average Precision (AP)** — среднее precision, взвешенное приростами recall при снижении порога

$$AP=\sum_i (R_i-R_{i-1})P_i,\qquad P_i=\frac{TP_i}{TP_i+FP_i},\quad R_i=\frac{TP_i}{TP_i+FN_i}$$

Здесь $i$ — номер порога при последовательном снижении, $P_i$ и $R_i$ — precision и recall при этом пороге, $R_0=0$. AP принимает значения от 0 до 1; больше — лучше

**ROC-AUC** — вероятность того, что кредит с дефолтом получит большую оценку риска, чем кредит без дефолта, с половинным вкладом при равных оценках. Значения от 0 до 1; больше — лучше, 0,5 соответствует случайному ранжированию

**Brier score** — средняя квадратичная ошибка прогноза вероятности

$$\mathrm{Brier}=\frac{1}{n}\sum_{j=1}^{n}(s_j-y_j)^2$$

Здесь $s_j$ — прогноз вероятности дефолта, $y_j\in\{0,1\}$ — фактический исход, $n$ — число кредитов. Значения от 0 до 1; меньше — лучше

##### Правила выбора и итоговой оценки

Основной критерий сравнения моделей — AP. Порог выбираем по F2, отдавая приоритет обнаружению дефолтов. Стоимости ошибок не заданы, финансовую выгоду не оцениваем

| Что оцениваем или выбираем | Правило |
| --- | --- |
| Модель | Наибольшая AP на validation |
| Порог | Наибольшая F2 на validation для выбранной модели; при равенстве — больший порог |
| Итоговое качество | На test рассчитываем AP, ROC-AUC, Brier, Accuracy, Precision, Recall, F1, F2 и Balanced accuracy, показываем TP, FP, FN и TN |

Базовый вариант для сравнения — **Dummy**: постоянный риск, равный доле дефолтов в train. Минимальное требуемое качество в задании не указано